## **Step-1: Import Libraries**

In [209]:
import os
import re

from langchain_classic.chains.qa_with_sources.map_reduce_prompt import question_prompt_template
from pyprojroot import here
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
#It is used to execute the whole query
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## **Step-2: Database Creation**

In [210]:
db_path=str(here("data")) + "/sqldb.db"
db_path_full=f"sqlite:///{db_path}"
sql_db=SQLDatabase.from_uri(db_path_full)
print(sql_db)

## **Step-3: LLM Creation**

In [211]:
os.environ["GITHUB TOKEN"]=""
token=os.environ.get("GITHUB TOKEN")
model_name="openai/gpt-4.1-mini"
endpoint="https://models.github.ai/inference"

if not token:
    raise ValueError("Please set GITHUB TOKEN environment variable")

llm_model=ChatOpenAI(
    model_name=model_name,
    openai_api_base=endpoint,
    openai_api_key=token
)

## **Step-4: Schema Creation**

In [212]:
sql_schema=sql_db.get_table_info()
schema=RunnablePassthrough.assign(sql_schema=lambda _:sql_schema)

## **Step-5: Query Prompt Creation**

In [213]:
sql_query_prompt=ChatPromptTemplate.from_template(
    """Given Database Schema Below.Write the SQL Query from user's question
    Database Schema:{sql_schema}
    User's Question:{question}
    SQL Query:"""
)

## **Step-6: Function Creation**

In [214]:
def sql_identifying_function(llm_response):
    markdown_code_blocks=re.search(r'```sql\s*(.*?)\s*```', llm_response, re.IGNORECASE|re.DOTALL)
    if markdown_code_blocks:
        sql_query=markdown_code_blocks.group(1).strip()
        return sql_query

    sql_label_format=re.search(r'SQLQuery:\s*(.*?)(?:\n|$)', llm_response, re.IGNORECASE)
    if sql_label_format:
        sql_query=sql_label_format.group(1).strip()
        return sql_query

    plain_query=re.match(r'^\s*(SELECT/DROP/CREATE/ALTER/DELETE/INSERT/UPDATE)\s+', llm_response, re.IGNORECASE|re.DOTALL)
    if plain_query:
        sql_query=plain_query
        return sql_query

    return None


## **Step-7: Execution Creation**

In [215]:
sql_execution_tool=QuerySQLDataBaseTool(db=sql_db)

## **Step-8: Chain Creation**

In [216]:
sql_query_chain=schema|sql_query_prompt|llm_model|StrOutputParser()|sql_identifying_function

In [217]:
sql_result_chain=sql_query_chain|sql_execution_tool

## **Step-9: Output Generation**

In [218]:
question=["How many employees are there",
          "Give me the names of all employees (don't use limit)",
          "Give me all employees title with no limit"]

In [219]:
sql_query=sql_query_chain.invoke(
    {
        "question":question[0]
    }
)

In [220]:
sql_result=sql_result_chain.invoke(
    {
        "question":question[0]
    }
)

## **Step-10: Human Answer Generation**


In [221]:
answer_prompt = ChatPromptTemplate.from_template(
    f"""You are a helpful database query assistant. Answer the user's question using only the SQL query result.
    User's Question:{question[0]}
    SQL Query:{sql_query}
    SQL Result:{sql_result}
    Give a natural language answer."""
)

In [222]:
answer_chain=answer_prompt|llm_model|StrOutputParser()

In [223]:
answer_input={
    "question":question[0],
    "sql_query":sql_query,
    "sql_result":sql_result
}

In [224]:
answer=answer_chain.invoke(answer_input)

## **Visualize the Final Output**

In [225]:
print("User Question:",question[0])
print("SQL Query:",sql_query)
print("SQL result:",sql_result)
print("Final Answer:",answer)

User Question: How many employees are there
SQL Query: SELECT COUNT(*) AS NumberOfEmployees
FROM Employee;
SQL result: [(8,)]
Final Answer: There are 8 employees.


## **All Output Together**

In [234]:
for i,q in enumerate(question,1):
    print(f"Question {i}:{q}")
    sql_query=sql_query_chain.invoke(
        {
            "question":q,
        }
    )
    sql_result=sql_result_chain.invoke(
        {
            "question":q,
        }
    )
    answer_prompt=ChatPromptTemplate.from_template(
        f"""You are a helpful database query assistant. Answer the user's question using only the SQL query result.
        User's Question:{q}
        SQL Query:{sql_query}
        SQL Result:{sql_result}
        Give a natural language answer."""
    )
    answer_chain=answer_prompt|llm_model|StrOutputParser()
    answer_input={
        "question":q,
        "sql_query":sql_query,
        "sql_result":sql_result
    }
    answer=answer_chain.invoke(answer_input)

    print("User Question:",q)
    print("SQL Query:",sql_query)
    print("SQL result:",sql_result)
    print("Final Answer:",answer)
    print("====================================================================")

Question 1:How many employees are there
User Question: How many employees are there
SQL Query: SELECT COUNT(*) AS EmployeeCount
FROM Employee;
SQL result: [(8,)]
Final Answer: There are 8 employees.
Question 2:Give me the names of all employees (don't use limit)
User Question: Give me the names of all employees (don't use limit)
SQL Query: SELECT FirstName, LastName
FROM Employee;
SQL result: [('Andrew', 'Adams'), ('Nancy', 'Edwards'), ('Jane', 'Peacock'), ('Margaret', 'Park'), ('Steve', 'Johnson'), ('Michael', 'Mitchell'), ('Robert', 'King'), ('Laura', 'Callahan')]
Final Answer: The names of all employees are Andrew Adams, Nancy Edwards, Jane Peacock, Margaret Park, Steve Johnson, Michael Mitchell, Robert King, and Laura Callahan.
Question 3:Give me all employees title with no limit
User Question: Give me all employees title with no limit
SQL Query: SELECT DISTINCT Title
FROM Employee;
SQL result: [('General Manager',), ('Sales Manager',), ('Sales Support Agent',), ('IT Manager',), ('

## **Show Each Step's Output**

In [226]:
sql_result_chain.invoke(
    {"question":"Give me the names of all employees in the company (dont use any limit)"}
)

"[('Andrew', 'Adams'), ('Nancy', 'Edwards'), ('Jane', 'Peacock'), ('Margaret', 'Park'), ('Steve', 'Johnson'), ('Michael', 'Mitchell'), ('Robert', 'King'), ('Laura', 'Callahan')]"

In [227]:
input={
    "question":"Give me all employee title with no limit"
}

In [228]:
schema_output=schema.invoke(input)
print(schema_output)

{'question': 'Give me all employee title with no limit', 'sql_schema': '\nCREATE TABLE "Album" (\n\t"AlbumId" INTEGER NOT NULL, \n\t"Title" NVARCHAR(160) NOT NULL, \n\t"ArtistId" INTEGER NOT NULL, \n\tPRIMARY KEY ("AlbumId"), \n\tFOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")\n)\n\n/*\n3 rows from Album table:\nAlbumId\tTitle\tArtistId\n1\tFor Those About To Rock We Salute You\t1\n2\tBalls to the Wall\t2\n3\tRestless and Wild\t2\n*/\n\n\nCREATE TABLE "Artist" (\n\t"ArtistId" INTEGER NOT NULL, \n\t"Name" NVARCHAR(120), \n\tPRIMARY KEY ("ArtistId")\n)\n\n/*\n3 rows from Artist table:\nArtistId\tName\n1\tAC/DC\n2\tAccept\n3\tAerosmith\n*/\n\n\nCREATE TABLE "Customer" (\n\t"CustomerId" INTEGER NOT NULL, \n\t"FirstName" NVARCHAR(40) NOT NULL, \n\t"LastName" NVARCHAR(20) NOT NULL, \n\t"Company" NVARCHAR(80), \n\t"Address" NVARCHAR(70), \n\t"City" NVARCHAR(40), \n\t"State" NVARCHAR(40), \n\t"Country" NVARCHAR(40), \n\t"PostalCode" NVARCHAR(10), \n\t"Phone" NVARCHAR(24), \n\t"Fax" NV

In [229]:
prompt_output=sql_query_prompt.invoke(schema_output)
print(prompt_output)

messages=[HumanMessage(content='Given Database Schema Below.Write the SQL Query from user\'s question\n    Database Schema:\nCREATE TABLE "Album" (\n\t"AlbumId" INTEGER NOT NULL, \n\t"Title" NVARCHAR(160) NOT NULL, \n\t"ArtistId" INTEGER NOT NULL, \n\tPRIMARY KEY ("AlbumId"), \n\tFOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")\n)\n\n/*\n3 rows from Album table:\nAlbumId\tTitle\tArtistId\n1\tFor Those About To Rock We Salute You\t1\n2\tBalls to the Wall\t2\n3\tRestless and Wild\t2\n*/\n\n\nCREATE TABLE "Artist" (\n\t"ArtistId" INTEGER NOT NULL, \n\t"Name" NVARCHAR(120), \n\tPRIMARY KEY ("ArtistId")\n)\n\n/*\n3 rows from Artist table:\nArtistId\tName\n1\tAC/DC\n2\tAccept\n3\tAerosmith\n*/\n\n\nCREATE TABLE "Customer" (\n\t"CustomerId" INTEGER NOT NULL, \n\t"FirstName" NVARCHAR(40) NOT NULL, \n\t"LastName" NVARCHAR(20) NOT NULL, \n\t"Company" NVARCHAR(80), \n\t"Address" NVARCHAR(70), \n\t"City" NVARCHAR(40), \n\t"State" NVARCHAR(40), \n\t"Country" NVARCHAR(40), \n\t"PostalCode" N

In [230]:
llm_output=llm_model.invoke(prompt_output)
print(llm_output)

content='```sql\nSELECT DISTINCT Title\nFROM Employee;\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 2186, 'total_tokens': 2198, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 2048}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 30, 'engine_ttlt_ms': 124, 'pre_inference_ms': 136, 'service_tbt_ms': 8, 'service_ttft_ms': 669, 'service_ttlt_ms': 726, 'total_duration_ms': 598, 'user_visible_ttft_ms': 533}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b7c8a4dc64', 'id': 'chatcmpl-E31Yt4aMkBORqDklNJPwmQPz7tVhZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f75e9-f534-7d80-9792-50ffa75c1c89-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 2186,

In [231]:
parser_output=StrOutputParser().invoke(llm_output.content)
print(parser_output)

```sql
SELECT DISTINCT Title
FROM Employee;
```


In [232]:
function_output=sql_identifying_function(parser_output)
print(function_output)

SELECT DISTINCT Title
FROM Employee;


In [233]:
tool_output=sql_execution_tool.invoke(function_output)
print(tool_output)

[('General Manager',), ('Sales Manager',), ('Sales Support Agent',), ('IT Manager',), ('IT Staff',)]
